In [6]:
import numpy as np
import pandas as pd

In [2]:
tuning_all = pd.read_csv('tuning-results.csv')
# tuning_binary = pd.read_csv("tuning-results-binary.csv")

In [3]:
best_lambdas_all = {}
for dataset in tuning_all['data_name'].unique():
    for prerank in tuning_all.prerank.unique():
        subdf = tuning_all[(tuning_all['data_name'] == dataset) & (tuning_all['prerank'] == prerank)]
        if len(subdf['lambda'].values) == 6:
            if 0.0 not in subdf['lambda'].values:
                continue
            baseline_energy = subdf[subdf['lambda'] == 0.0]['energy'].values[0]
            valid = subdf[subdf['energy'] <= 1.1 * baseline_energy]
            best = valid.sort_values('pce').iloc[0] if not valid.empty else subdf.sort_values('pce').iloc[0]
            best_lambdas_all[(dataset, prerank)] = best['lambda'] 
        else: print(f"not enough lambdas for {dataset} {prerank}")

not enough lambdas for births1 dependency
not enough lambdas for wage dependency
not enough lambdas for meps_21 dependency
not enough lambdas for meps_19 dependency
not enough lambdas for meps_20 dependency
not enough lambdas for house dependency
not enough lambdas for bio dependency
not enough lambdas for blog_data dependency
not enough lambdas for calcofi dependency
not enough lambdas for ansur2 dependency
not enough lambdas for taxi marginal
not enough lambdas for taxi mean
not enough lambdas for taxi variance
not enough lambdas for taxi dependency
not enough lambdas for taxi pca
not enough lambdas for taxi density
not enough lambdas for taxi cdf


In [ ]:
best_lambdas_all

In [5]:
best_lambdas_all[('calcofi', 'cdf')]

np.float64(5.0)

In [26]:
df = pd.read_csv('metrics-after-reg-ansur-taxi.csv')
df.head()

,data_name,seed,prerank,pce,nll,energy,mse
0,ansur2,0,marginal,0.029722,1.895472,0.550301,0.365455
1,ansur2,42,marginal,0.022534,1.858149,0.535462,0.360347
2,ansur2,866,marginal,0.035830,1.832666,0.548105,0.395083
3,ansur2,12,marginal,0.047822,1.922425,0.554950,0.382537
4,ansur2,4,marginal,0.026563,1.833968,0.529114,0.350377


In [27]:
df.shape

(70, 7)

In [28]:
agg_df = (
     df.groupby(["data_name", "prerank"])
      .agg(pce_mean=("pce", "mean"),
           pce_se=("pce", lambda x: x.std() / (len(x) ** 0.5)),
           nll_mean=("nll", "mean"),
           nll_se=("nll", lambda x: x.std() / (len(x) ** 0.5)),
           energy_mean=("energy", "mean"),
           energy_se=("energy", lambda x: x.std() / (len(x) ** 0.5)),
           mse_mean=("mse", "mean"),
           mse_se=("mse", lambda x: x.std() / (len(x) ** 0.5)))
          ).reset_index()

In [6]:
agg_df[agg_df['data_name']=='births1']

,data_name,prerank,pce_mean,pce_se,nll_mean,nll_se,energy_mean,energy_se,mse_mean,mse_se
13,births1,cdf,0.026634,0.001301,1.391951,0.077995,0.729287,0.004198,0.860039,0.009327
14,births1,density,0.034024,0.003453,2.120752,0.041119,0.711908,0.003201,0.840337,0.006422
15,births1,dependency,0.027086,0.001969,1.629785,0.119460,0.714112,0.006299,0.852112,0.011010
16,births1,marginal,0.028327,0.000917,1.820428,0.174687,0.724457,0.005878,0.852920,0.008848
17,births1,mean,0.026544,0.001660,1.372964,0.083237,0.733607,0.007451,0.873195,0.015179
18,births1,pca,0.034281,0.000435,1.363047,0.092498,0.729912,0.004354,0.868853,0.011757
19,births1,variance,0.031463,0.002083,1.738637,0.196139,0.718074,0.005310,0.858125,0.013846


In [ ]:
dataset = "births1"
subdf = agg_df[agg_df["data_name"] == dataset]
# for _, row in subdf.iterrows():
#     print(f"""{row['prerank']}: {row['nll_mean']:.3f} ({row['nll_se']:.3f}) & {row['energy_mean']:.3f} ({row['energy_se']:.3f}) & {row['mse_mean']:.3f} ({row['mse_se']:.3f})""")

,data_name,prerank,pce_mean,pce_se,nll_mean,nll_se,energy_mean,energy_se,mse_mean,mse_se
13,births1,cdf,0.026634,0.001301,1.391951,0.077995,0.729287,0.004198,0.860039,0.009327
14,births1,density,0.034024,0.003453,2.120752,0.041119,0.711908,0.003201,0.840337,0.006422
15,births1,dependency,0.027086,0.001969,1.629785,0.119460,0.714112,0.006299,0.852112,0.011010
16,births1,marginal,0.028327,0.000917,1.820428,0.174687,0.724457,0.005878,0.852920,0.008848
17,births1,mean,0.026544,0.001660,1.372964,0.083237,0.733607,0.007451,0.873195,0.015179
18,births1,pca,0.034281,0.000435,1.363047,0.092498,0.729912,0.004354,0.868853,0.011757
19,births1,variance,0.031463,0.002083,1.738637,0.196139,0.718074,0.005310,0.858125,0.013846


In [37]:
order = ["marginal", "mean", "variance", "dependency", "pca", "density", "cdf"]
subset = agg_df[agg_df["data_name"] == "taxi"]
subset = subset.set_index("prerank").loc[order]
result = " & ".join(f"{row['mse_mean']:.3f} ({row['mse_se']:.3f})" for _, row in subset.iterrows())
print(result)


0.865 (0.013) & 0.833 (0.010) & 0.881 (0.013) & 0.842 (0.009) & 0.857 (0.009) & 0.832 (0.010) & 0.842 (0.009)
